# ai-detector — HUẤN LUYỆN mô hình phát hiện giọng giả

Notebook **tự chứa toàn bộ mã nguồn** (38 file, 83 KB nhúng sẵn) —
không cần clone repo, không cần dataset chứa code. Import lên Kaggle là chạy được.

```
corpus (real + fake) ──> augmentation ──> WavLM ──> Classifier ──> REAL / FAKE
```

## Pipeline nằm ở HAI notebook

| Notebook | Làm gì | Cần gì trong Input |
|---|---|---|
| `aidetector_dataset.ipynb` | ingest → generate → kiểm tra → đẩy lên Dataset | một bộ giọng thật (VIVOS, Common Voice vi…) |
| **`aidetector_train.ipynb`** ← file này | split → augment → WavLM → classifier → đánh giá | corpus do file kia đẩy lên |

Cả hai nhúng **cùng một payload mã nguồn** và dùng **cùng ô A1b** để nạp corpus, nên
không có chuyện huấn luyện trên một chuẩn dữ liệu khác chuẩn lúc sinh. Mọi ô trong file
này đều thuộc việc huấn luyện — **Save & Run All** là đúng.

File này **không sinh audio và không đẩy gì lên dataset corpus**: phần B chạy `augment`,
đẩy sau đó là bơm dữ liệu phái sinh vào dataset, buộc mọi phiên sau tải thêm phần mà một
lệnh `augment` sinh lại được trong vài phút. Mô hình và báo cáo đi đường Output — ô B4
gói `model.zip` và `reports_bundle.zip`.

## Cần bật trong panel bên phải

| Mục | Đặt thành | Vì sao |
|---|---|---|
| **Accelerator** | `GPU T4 x2` hoặc `P100` | WavLM trích đặc trưng cho hàng chục nghìn clip |
| **Internet** | `On` | tải checkpoint WavLM, cài thư viện |

Rồi **Add Input → Datasets → corpus đã sinh** (`DATASET_ID` ở ô setup, mặc định
`sonpham12/vivos-fake-v2`). Không nạp được corpus thì ô A1b **dừng ngay** — huấn luyện
trên tay không là bỏ cả phiên GPU.

## 0. Chuẩn bị

In [ ]:
# Toàn bộ package aidetector + configs, nén tar.gz rồi base64.
# sha256(payload) = 2a34a5f0985c251b…
_PAYLOAD = (
    "..."
    "vfr36t+rf6/+vfr36t+rf+bf/w9HVRazALAEAA=="
)

import base64, hashlib, importlib, io, os, shutil, sys, tarfile
from pathlib import Path

_raw = base64.b64decode(_PAYLOAD)
assert hashlib.sha256(_raw).hexdigest() == "2a34a5f0985c251b8d248afd9391b9499a07eb3eaa0f4002a08fa50628229cee", "payload hỏng khi sao chép notebook"

WORK = Path("/kaggle/working/ai-detector")
WORK.mkdir(parents=True, exist_ok=True)

# Xoá sạch cây mã nguồn cũ trước khi bung: chạy đè lên bản cũ sẽ để sót những file
# đã bị bỏ ở bản mới, và để lại __pycache__ cũ.
for _old in ("aidetector", "configs"):
    shutil.rmtree(WORK / _old, ignore_errors=True)

with tarfile.open(fileobj=io.BytesIO(_raw), mode="r:gz") as _tf:
    try:
        _tf.extractall(WORK, filter="data")     # Python >= 3.12
    except TypeError:
        _tf.extractall(WORK)

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))

# Kernel Kaggle sống xuyên suốt nhiều lần chạy. Nếu phiên trước đã import
# aidetector, Python giữ nguyên module cũ trong sys.modules và lờ đi mã vừa bung —
# biểu hiện là những lỗi rất khó hiểu kiểu "cannot import name X" dù X có trong
# file. Phải gỡ chúng ra để lần import sau đọc lại từ đĩa.
_stale = [m for m in sys.modules if m == "aidetector" or m.startswith("aidetector.")]
for _m in _stale:
    del sys.modules[_m]
importlib.invalidate_caches()

CFG = "configs/kaggle.yaml"


# Chạy một stage của pipeline và DỪNG notebook ngay nếu nó lỗi.
# Không dùng `!python -m aidetector ...`: trong Jupyter, lệnh shell lỗi vẫn để
# notebook chạy tiếp các ô sau, nên một stage hỏng sẽ âm thầm kéo theo cả loạt lỗi
# vô nghĩa ở dưới — hoặc tệ hơn, chạy tiếp trên dữ liệu cũ còn sót lại.
#
# `optional=True` dành cho bước không bắt buộc (vd một engine sinh fake cần GPU
# hoặc cần quyền tải checkpoint): hỏng thì báo rồi đi tiếp, vì dữ liệu đã có từ
# các bước trước vẫn dùng được.
def run(*args, optional=False):
    import subprocess

    # Chuẩn audio phải giống nhau ở MỌI stage. Chỉ hạ min_seconds cho `ingest` mà không
    # hạ cho `generate` là real được giữ tới 2s trong khi fake dưới 3s bị bỏ — chính độ
    # dài thành dấu hiệu phân biệt hai lớp, đúng thứ chuỗi chuẩn hoá này tồn tại để bịt.
    chuan = []
    for _k, _v in (("min_seconds", globals().get("MIN_SECONDS")),
                   ("max_seconds", globals().get("MAX_SECONDS"))):
        if _v:
            chuan += ["--set", f"audio.{_k}={_v}"]

    cmd = [sys.executable, "-m", "aidetector", *[str(a) for a in args], *chuan, "-c", CFG]
    print("$ python -m aidetector " + " ".join(str(a) for a in [*args, *chuan])
          + f" -c {CFG}\n")
    if subprocess.run(cmd).returncode == 0:
        return True
    if optional:
        print(f"\n⚠ Bước tuỳ chọn {args[0]!r} không chạy được — bỏ qua, đi tiếp.")
        return False
    raise SystemExit(f"✖ Stage {args[0]!r} thất bại — xem log ngay phía trên, "
                     f"đừng chạy tiếp các ô sau.")


print(f"Đã bung {len(_raw) / 1024:.0f} KB mã nguồn vào {WORK}")
if _stale:
    print(f"Đã gỡ {len(_stale)} module aidetector cũ khỏi bộ nhớ kernel")

Cài thư viện — lượt cài **nhẹ nhất** trong hai notebook: không engine sinh audio nào cả,
đỡ vài phút và tránh hẳn màn giằng nhau về phiên bản `transformers` (Kokoro cần 4.x,
OmniVoice cần ≥5.3). WavLM chạy được trên cả hai nhánh nên cứ dùng bản Kaggle cài sẵn.

`TTS_ENGINES` vẫn có trong ô dưới vì hai file dùng chung đúng một ô setup, nhưng ở file
này nó không có tác dụng gì.

In [ ]:
# File này chạy phần B — huấn luyện. Phần còn lại ở
# aidetector_dataset.ipynb — cùng payload, cùng ô A1b.
MODE = "train"

# Kho dữ liệu dùng chung cho MỌI chế độ: phần A đẩy corpus lên đây, mọi phiên sau nạp
# lại từ đây. Khai báo một chỗ duy nhất — A1b (nạp) và A2b (đẩy) đều đọc biến này, để
# không bao giờ có chuyện đẩy lên một dataset mà nạp về từ một dataset khác.
DATASET_ID = "sonpham12/vivos-fake-v2"

# Ngưỡng độ dài tối thiểu của một clip, áp cho CẢ real và fake ở mọi stage (ô `run`
# ở trên tự dán `--set audio.min_seconds` vào từng lệnh).
#
# Số đo thật trên VIVOS: ở 3.0 giữ 8.246/12.421 clip (66%), bỏ 4.175 vì quá ngắn — trong
# đó 2.865 clip vẫn dài ≥2s. Hạ xuống 2.0 lấy lại chừng đó, tức corpus ~11.100 và thêm
# khoảng 3 giờ sinh. Đổi lại mỗi clip mang ít bằng chứng hơn cho mô hình.
#
# ĐỪNG đổi `short_policy` sang "pad": real bị đệm im lặng trong khi fake (~4s) thì không
# — đó là tự tạo ra dấu hiệu phân biệt hai lớp.
MIN_SECONDS = 3.0
# Độ dài tối đa. `ingest` cắt bản thu dài hơn mức này thành các đoạn ĐÚNG độ dài đó, đánh
# số trong thư mục của bản thu; đoạn cuối ngắn hơn MIN_SECONDS thì bỏ. Nên đây cũng là
# nút để biến một file 60 giây thành 15 đoạn 4 giây, không cần code cắt riêng.
MAX_SECONDS = 10.0

# Piper/Kokoro đang TẮT: giọng cố định, mô hình bắt ở EER 0.00% nên không dạy được gì,
# chỉ làm loãng dataset. Bật lại bằng: TTS_ENGINES = ["piper", "kokoro"]
TTS_ENGINES = []

# Hai giá trị, một cho mỗi file — không còn "both": phần A và phần B nằm ở hai notebook,
# nên "một phiên chạy cả hai" là chuyện không tồn tại nữa. Vẫn kiểm, vì MODE sai mà chạy
# tiếp im lặng là bỏ cả phiên GPU.
if MODE not in ("dataset", "train"):
    raise SystemExit(f'MODE={MODE!r} không hợp lệ — "dataset" hoặc "train".')
MAKE_DATASET = MODE == "dataset"
DO_TRAIN = MODE == "train"

# Nói rõ vì sao một ô không làm gì: Run All mà im lặng thì log không đọc được.
def skipped(what):
    print(f"⏭ MODE={MODE!r} — bỏ qua {what}.")

!apt-get -qq install -y ffmpeg > /dev/null 2>&1 || true   # cần cho augment MP3/AAC

# subprocess chứ không `!pip`: magic của IPython không lồng vào `if` được.
import subprocess
import sys

def pip(*args, ok_to_fail=False):
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args]).returncode:
        if not ok_to_fail:
            raise SystemExit(f"pip install {' '.join(args)} thất bại — xem log phía trên")
        print(f"⚠ bỏ qua: pip install {' '.join(args)}")

pip("-r", "requirements.txt")
# Image Kaggle đang có kaggle 2.0.2 (log phiên trước tự cảnh báo). Bản đó có thể chưa
# biết token kiểu mới `KGAT_`, mà đó lại là đường xác thực để đẩy dataset.
pip("-U", "kaggle", ok_to_fail=True)
if not MAKE_DATASET:
    # Không sinh audio thì không cần engine nào. WavLM chạy được trên cả hai nhánh
    # transformers nên cứ để bản Kaggle cài sẵn — đây là chế độ cài nhẹ nhất.
    print("Chỉ huấn luyện — không cài engine sinh audio.")
elif TTS_ENGINES:
    pip("piper-tts", ok_to_fail=True)
    pip("git+https://github.com/iamdinhthuan/Kokoro-Vietnamese.git", ok_to_fail=True)
    pip("transformers>=4.48,<5")
else:
    # Không có Kokoro thì bỏ được màn ghim-rồi-nâng transformers giữa phiên.
    pip("omnivoice", "transformers>=5.3")

import transformers, torch
print(f"MODE: {MODE} · phần A {'BẬT' if MAKE_DATASET else 'tắt'}"
      f" · phần B {'BẬT' if DO_TRAIN else 'tắt'}")
print(f"TTS: {TTS_ENGINES or 'tắt — chỉ voice cloning'}")
print(f"transformers {transformers.__version__} · torch {torch.__version__} "
      f"· CUDA {torch.cuda.is_available()}")

In [ ]:
run("info")

### A1b. Nạp corpus của phiên trước

Bung `DATASET_ID` (khai báo ở ô setup) ra `/kaggle/working` để chạy tiếp. `ingest` và
`generate` đều idempotent theo `utt_id` nên chúng chỉ làm phần còn thiếu — không có bước
nào làm lại từ đầu.

Muốn nối lại thì phải **Add Input → Datasets → dataset đó**. Chưa add thì ô này vẫn hỏi
Kaggle xem dataset đang có gì (nếu đã cài token) rồi nhắc — chứ không im lặng bắt đầu lại
từ đầu và làm mất công phiên trước. Mount nhiều dataset thì ô này lấy **đúng** cái khớp
`DATASET_ID`, không phải cái đầu bảng chữ cái.

Ô này **giống nhau từng byte ở cả hai notebook** — nó là đường duy nhất mang corpus vào
một phiên. Khác nhau chỉ ở chỗ thiếu corpus thì sao: notebook dataset bắt đầu từ đầu,
còn notebook train dừng ngay, kể cả khi corpus bung ra được nhưng thiếu hẳn một lớp —
huấn luyện trên tay không là bỏ cả phiên GPU.

#### `corpus.zip` không còn trên dataset là chuyện BÌNH THƯỜNG

Kaggle **tự giải nén** mọi `.zip` đưa lên dataset và không giữ lại bản nén. Nên
`corpus.zip` mà A2b đẩy lên biến thành cây `<bộ>/real/ <bộ>/fake/ <bộ>/metadata.csv`
nằm thẳng trong mount. Ô này nhận cả hai dạng:

| Mount có gì | Ô này làm gì |
|---|---|
| `corpus.zip` | `unpack` như cũ |
| cây `<bộ>/real/ <bộ>/fake/` đã bung | **symlink** vào `/kaggle/working/corpus` — không copy |
| chỉ `metadata.csv`, không audio | DỪNG, in ra đang mount gì để soi |

Đường symlink còn nhanh hơn zip: khỏi mất vài phút bung và 1 GB đĩa. `/kaggle/input`
chỉ-đọc, nên chỉ `metadata.csv` được copy thật (split ghi cột `split`, validate ghi
`checked` vào đó); audio cũ là symlink trỏ vào mount, audio mới ghi thẳng vào cây.

Corpus **tách theo bộ dữ liệu** nên có nhiều `metadata.csv` — mỗi bộ một file. Ô này copy
tất cả, giữ đúng vị trí tương đối của từng file. Cột `path` tính từ gốc corpus ở cả cấu
trúc mới và cấu trúc gộp cũ, nên nó tự nhận ra gốc là thư mục chứa manifest hay thư mục
cha của nó — không phải khai gì.

In [ ]:
import glob
import subprocess
from pathlib import Path

CORPUS = Path("/kaggle/working/corpus")

# Kaggle mount dataset ở /kaggle/input/<slug>. Tìm ĐÚNG dataset đã cấu hình trước rồi
# mới chấp nhận corpus.zip bất kỳ: mount nhiều dataset mà "lấy cái cuối theo abc" thì
# phiên này nối tiếp công của dataset nào là chuyện xổ số.
def _find(name):
    slug = DATASET_ID.split("/")[-1]
    return (sorted(glob.glob(f"/kaggle/input/{slug}/**/{name}", recursive=True))
            or sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)))

# Corpus tách theo BỘ DỮ LIỆU: mỗi bộ một thư mục với `metadata.csv` của riêng nó. Nên
# "corpus có gì chưa" là câu hỏi về một DANH SÁCH file, không phải một file.
#
# Vẫn nhận manifest gộp ở gốc (cấu trúc cũ) và tên cũ `manifest.csv`: corpus đã đẩy lên
# Kaggle ở các phiên trước dùng chúng, bỏ đọc là vứt luôn hàng giờ GPU đã trả.
def _cac_meta(thu_muc):
    thu_muc = Path(thu_muc)
    if not thu_muc.is_dir():
        return []
    ra = []
    for goc in (thu_muc, *sorted(p for p in thu_muc.iterdir() if p.is_dir())):
        for ten in ("metadata.csv", "manifest.csv"):
            if (goc / ten).exists():
                ra.append(goc / ten)
                break
    return ra

_mounted = _find("corpus.zip")
# `or` chứ không phải `+`: có cả hai tên thì phải lấy bản MỚI, mà `_loose[-1]` ở dưới
# lấy phần tử cuối — nối danh sách lại là chọn đúng bản cũ.
_loose = _find("metadata.csv") or _find("manifest.csv")

# Kaggle GIẢI NÉN mọi .zip đưa lên dataset và KHÔNG giữ lại bản nén. Nên `corpus.zip`
# vừa đẩy lên biến thành cây `<bộ>/real/ <bộ>/fake/ <bộ>/metadata.csv` trong mount, và
# "không thấy corpus.zip" hầu như chưa bao giờ là mất dữ liệu — dữ liệu ở đó, bung sẵn.
#
# Cây bung sẵn còn nạp NHANH HƠN zip: đọc trực tiếp từ /kaggle/input, khỏi mất vài phút
# bung và 1 GB đĩa. Nhưng mount chỉ-đọc, mà mọi stage sau (generate, augment, split,
# validate) đều ghi vào corpus — nên phải dựng một cây GHI ĐƯỢC ở /kaggle/working/corpus:
# metadata.csv là bản copy, mỗi audio cũ là một symlink trỏ vào mount, audio mới ghi
# thẳng vào cây như thường.
# Các cây corpus đã bung trong mount, tốt nhất trước: (tỉ lệ khớp, số dòng, gốc, meta).
# "Khớp" = manifest kể tên audio nào thì audio đó có mặt cạnh nó. Đó là phép duy nhất
# phân biệt được gốc corpus thật với bản metadata.csv để rời ngoài zip — hai file trùng
# nội dung, chỉ khác chỗ đứng.
def _cay_bung_san():
    import csv

    uv = {}
    for duong in _find("metadata.csv") + _find("manifest.csv"):
        duong = Path(duong)
        with open(duong, encoding="utf-8", newline="") as fh:
            rows = list(csv.DictReader(fh))
        if not rows:
            continue
        # Cột `path` tính từ GỐC CORPUS ở cả hai cấu trúc, nên gốc là thư mục chứa
        # manifest (bảng gộp cũ) HOẶC thư mục cha của nó (manifest của một bộ). Thử cả
        # hai rồi lấy cái khớp hơn — đó là phép duy nhất phân biệt được hai trường hợp.
        #
        # Đếm trên mẫu 200 dòng: stat 15 nghìn file qua mount là chậm thật, mà tỉ lệ
        # khớp thì mẫu đã nói đủ — cây đúng khớp gần 100%, cây sai khớp gần 0%.
        mau = rows[:: max(1, len(rows) // 200)][:200]
        for goc in (duong.parent, duong.parent.parent):
            khop = sum(1 for r in mau if r.get("path") and (goc / r["path"]).exists())
            # Gộp theo GỐC, không theo manifest: một corpus tách bộ có nhiều manifest
            # nhưng chỉ một gốc, và nó phải được tính là một ứng viên với đủ số dòng.
            ti, tong = uv.get(str(goc), (0.0, 0))
            if khop:
                uv[str(goc)] = (max(ti, khop / len(mau)), tong + len(rows))
    ra = [(ti, tong, Path(goc)) for goc, (ti, tong) in uv.items()]
    ra.sort(reverse=True)
    return ra

# Dựng corpus ghi được từ cây chỉ-đọc: manifest copy, audio symlink.
def _muon_cay(goc):
    import csv
    import os
    import shutil

    CORPUS.mkdir(parents=True, exist_ok=True)
    xong = thieu = 0
    # MỌI manifest của gốc đó — corpus tách theo bộ thì mỗi bộ một file. Mỗi file được
    # copy về đúng vị trí tương đối của nó, vì đó là chỗ `Manifest` sẽ tìm.
    for meta in _cac_meta(goc):
        # Manifest phải là bản COPY: split ghi cột `split` vào nó, validate ghi `checked`.
        dich_meta = CORPUS / meta.relative_to(goc)
        dich_meta.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(meta, dich_meta)
        with open(meta, encoding="utf-8", newline="") as fh:
            for row in csv.DictReader(fh):
                if not row.get("path"):
                    thieu += 1
                    continue
                nguon, dich = goc / row["path"], CORPUS / row["path"]
                if dich.exists():
                    continue
                if not nguon.exists():
                    thieu += 1
                    continue
                dich.parent.mkdir(parents=True, exist_ok=True)
                os.symlink(nguon, dich)
                xong += 1
    return xong, thieu

# Trạng thái tường minh do phiên trước ghi lại: xong tới speaker nào. Vài KB, đọc được
# ngay trên trang dataset, và không phải suy ra từ manifest hàng nghìn dòng.
_tt = _find("progress.json")
if _tt:
    import json as _json

    _s = _json.loads(Path(_tt[-1]).read_text(encoding="utf-8"))
    print(f"Trạng thái phiên trước ghi lại: {_s['targets_done']}/{_s['targets_total']}"
          f" khuôn đã có fake · speaker {len(_s['speakers_done'])} xong"
          f" · {len(_s['speakers_partial'])} dở dang"
          f" · {len(_s['speakers_todo'])} chưa động tới")
    # Theo từng NGUỒN: bộ dữ liệu nào đã nằm trên kho và đã duyệt tới đâu. Nguồn đã có
    # đủ thì phiên này không phải chuẩn hoá lại cũng không phải soi lại — `ingest` bỏ qua
    # theo utt_id, `validate` bỏ qua theo dấu đã duyệt.
    for _ten, _o in sorted(_s.get("by_source", {}).items()):
        print(f"  nguồn {_ten:<22} real {_o['real']:>6} · fake {_o['fake']:>6}"
              f" · đã duyệt {_o['approved']:>6}")

if _cac_meta(CORPUS):
    print("Corpus đã có sẵn trong /kaggle/working — không bung đè lên.")
    run("info")
elif _mounted:
    print(f"Bung corpus từ {_mounted[-1]}")
    run("unpack", _mounted[-1])
elif _cay := next((u for u in _cay_bung_san() if u[0] >= 0.9), None):
    # Ngưỡng 0.9 chứ không phải 1.0: manifest luôn mới hơn ảnh chụp một nhịp, nên vài
    # bản ghi cuối chưa kịp có file là chuyện thường — `prune_missing` loại chúng ở dưới.
    _ti, _tong, _goc = _cay
    print(f"Không có corpus.zip — Kaggle đã giải nén nó. Dùng cây bung sẵn: {_goc}")
    _xong, _thieu = _muon_cay(_goc)
    print(f"Đã trỏ {_xong} audio vào {CORPUS} bằng symlink (không copy, không tốn đĩa)"
          + (f" · {_thieu} bản ghi chưa có file" if _thieu else ""))

    from aidetector.corpus.manifest import Manifest

    _m0 = Manifest.load(CORPUS, required=True)
    if _m0.prune_missing():
        _m0.save()
else:
    # DỪNG HẲN nếu dataset đã có dữ liệu mà phiên này không nạp được. Đi tiếp nghĩa là
    # ingest lại từ đầu rồi đẩy một corpus 0 fake ĐÈ LÊN công của các phiên trước —
    # `datasets version` là ảnh chụp toàn bộ thư mục, không phải cộng dồn.
    _co_du_lieu = ""
    if _loose:
        # manifest để rời ngoài zip chính là để đọc tiến độ mà không phải tải cả GB.
        import csv

        with open(_loose[-1], encoding="utf-8") as fh:
            rows = list(csv.DictReader(fh))
        fakes = [r for r in rows if r.get("label") == "fake" and not r.get("augment")]
        print(f"Thấy manifest của dataset: {len(rows)} bản ghi · {len(fakes)} fake"
              f" · {len({r['speaker'] for r in fakes})} speaker đã có fake")
        # Manifest có mà audio thì không: in ra ĐANG MOUNT GÌ, vì đó là thứ duy nhất
        # phân biệt "add sai dataset" với "version mới còn đang xử lý trên Kaggle".
        _goc_in = Path("/kaggle/input")
        _cac = sorted(d.name for d in _goc_in.iterdir()) if _goc_in.is_dir() else []
        print(f"Đang mount: {', '.join(_cac) or '(chưa add Input nào)'}")
        for _t, _n, _g in _cay_bung_san()[:3]:
            print(f"  {_g}: {_n} bản ghi · {100 * _t:.0f}% audio có mặt cạnh manifest")
        _co_du_lieu = (f"{len(rows)} bản ghi ({len(fakes)} fake), nhưng mount KHÔNG có"
                       " corpus.zip lẫn cây audio bung sẵn")
    else:
        # Chưa mount thì vẫn hỏi API cho biết dataset đang có gì.
        r = subprocess.run(["kaggle", "datasets", "files", DATASET_ID],
                           capture_output=True, text=True)
        if r.returncode == 0:
            print("Dataset trên Kaggle đang có:")
            print(r.stdout.strip()[:800])
            if any(t in r.stdout for t in ("corpus.zip", "metadata.csv",
                                           "manifest.csv", "progress.json")):
                _co_du_lieu = "dữ liệu trên dataset nhưng chưa Add Input"
        else:
            print("Chưa nối được tới dataset (chưa add Input, chưa có token, hoặc dataset trống).")

    if _co_du_lieu:
        raise SystemExit(
            f"DỪNG: dataset {DATASET_ID} đã có {_co_du_lieu}.\n"
            "Add Input → Datasets → dataset đó rồi chạy lại ô này.\n"
            "Chạy tiếp mà không nạp được là ingest lại từ đầu rồi ĐÈ MẤT công phiên trước."
        )
    if MAKE_DATASET:
        print("Dataset trống — phiên này bắt đầu từ đầu.")

# Nguồn nào đã nằm trong kho, đếm theo bản ghi REAL. Ô convert hỏi đúng dict này để
# quyết định có phải convert lại hay không — đọc từ manifest local (đã bung ở trên) chứ
# không từ progress.json, vì manifest luôn có còn progress.json thì version cũ có thể thiếu.
NGUON_DA_CO = {}

# Đã tới đâu rồi — con số này là mốc của cả phiên: phần A biết còn phải sinh bao nhiêu,
# phần B biết mình sắp huấn luyện trên cái gì.
if _cac_meta(CORPUS):
    from aidetector.corpus.manifest import Manifest

    _m = Manifest.load(CORPUS, required=True)
    for _r in _m:
        if not _r.augment and not _r.is_fake:
            NGUON_DA_CO[_r.source] = NGUON_DA_CO.get(_r.source, 0) + 1
    _done = len({f.speaker for f in _m.fakes})
    print(f"\nCorpus đang có: {len(_m.reals)} real · {len(_m.fakes)} fake"
          f" · {_done}/{len(_m.speakers('real'))} speaker đã có fake")

    # ĐÃ GEN ĐẾN ĐÂU so với đích "mỗi real đủ điều kiện có một fake". Đây là câu duy
    # nhất đáng hỏi trước khi bắt đầu một phiên nối tiếp, và nó đọc được từ chính
    # manifest — không cần nạp engine, không cần GPU.
    from aidetector.config import Config
    from aidetector.generate.texts import is_usable

    _c = Config.load(CFG)
    _pool = [r for r in _m.reals if not r.augment and r.text and is_usable(
        r.text, int(_c.get("generate.min_words", 6)), int(_c.get("generate.max_words", 40)))]
    _co_fake = {f.ref_utt_id for f in _m.fakes}
    _xong = sum(1 for r in _pool if r.utt_id in _co_fake)
    _con = len(_pool) - _xong
    print(f"Tiến độ gen   : {_xong}/{len(_pool)} real đủ điều kiện đã có fake"
          f" ({100 * _xong / max(len(_pool), 1):.0f}%) · còn {_con} mẫu"
          f" ≈ {_con * 3.7 / 3600:.1f} giờ trên T4")
    # Chỉ-huấn-luyện thì corpus không phải tiện lợi mà là điều kiện sống.
    if not MAKE_DATASET and not (_m.reals and _m.fakes):
        raise SystemExit(f"Corpus chỉ có một lớp (real={len(_m.reals)}, fake={len(_m.fakes)})"
                         " — phân loại real/fake cần cả hai.")
elif not MAKE_DATASET:
    raise SystemExit(
        f"MODE={MODE!r} nhưng không bung được corpus nào — không có gì để huấn luyện.\n"
        f"Add Input → Datasets → {DATASET_ID} rồi chạy lại ô này."
    )

### A1d. Dọn corpus cũ về cây hiện hành

Corpus bung ra từ phiên trước có thể còn cây cũ (`audio/<label>/…/<utt_id>.wav`). `migrate`
dời file về đúng chỗ và giữ nguyên `utt_id`, nên **không sinh lại gì**.

Idempotent, và chịu được ngắt giữa chừng: manifest chỉ lưu sau khi dời xong, phép cấp số
là tất định, nên chạy lại tính ra đúng những đường dẫn cũ và nhận lại phần đã dời.

In [ ]:
run("migrate")

---
# PHẦN B — Huấn luyện

Corpus đã được nạp ở ô **A1b** phía trên nên phần này không phải bung lại gì. Muốn lấy
corpus từ một dataset khác thì đổi `DATASET_ID` ở ô setup, hoặc gọi thẳng
`run("unpack", "/kaggle/input/<tên-dataset>/corpus.zip")`.

Kiểm tra chất lượng corpus (nghe thử, đo độ giống giọng, đo phát âm) nằm ở
`aidetector_dataset.ipynb` mục A4 — nó thuộc lúc **quyết định** dataset, không thuộc lúc
huấn luyện. Thành phần corpus thì A1b vừa in ở trên.

## B1. Chia tập → augment

`split` chạy **trước** `augment`: bản augment chỉ sinh cho train và bám đúng split
của bản gốc, còn val/test giữ audio sạch để số đo phản ánh dữ liệu thật. Chia
speaker-disjoint nên không có speaker nào xuất hiện ở hai tập.

Thêm `--holdout omnivoice` nếu muốn giữ hẳn một engine riêng cho test — đó là phép
đo sát thực tế nhất: mô hình có bắt được engine **chưa từng thấy** hay không.

In [ ]:
if DO_TRAIN:
    run("split")
    run("augment", "--copies", 1)
else:
    skipped("split + augment")

## B2. WavLM → Classifier

Embedding cache theo `utt_id` nên chạy lại chỉ trích phần mới. Đổi backbone chỉ cần
`--set features.backbone.name=wav2vec2` — cache tách riêng, không đè lên nhau.

In [ ]:
if DO_TRAIN:
    run("features")
    run("train")
    run("evaluate")
else:
    skipped("features + train + evaluate")

## B3. Kết quả

In [ ]:
if DO_TRAIN:
    import json
    from pathlib import Path
    from IPython.display import Image, display

    metrics = json.loads(Path("/kaggle/working/reports/metrics.json").read_text())
    overall = metrics["overall"]
    print(f"EER      : {overall['eer'] * 100:.2f}%      ← số đo chính")
    print(f"ROC-AUC  : {overall['roc_auc']:.4f}")
    print(f"min-DCF  : {overall['min_dcf']:.4f}")
    print(f"Accuracy : {overall['accuracy'] * 100:.2f}%  (ngưỡng {overall['threshold']:.3f})")

    print("\nTheo từng generator:")
    for name, entry in metrics["by_generator"].items():
        if "eer_vs_all_real" in entry:
            print(f"  {name:<42} n={entry['n']:>5} · EER {entry['eer_vs_all_real'] * 100:6.2f}%"
                  f" · bắt được {entry['detection_rate'] * 100:5.1f}%")
        elif "false_alarm_rate" in entry:
            print(f"  {name:<42} n={entry['n']:>5} · báo nhầm {entry['false_alarm_rate'] * 100:5.1f}%")

    print("\nClean vs augmented:")
    for name, entry in metrics["by_condition"].items():
        print(f"  {name:<12} n={entry['n']:>5} · điểm trung bình {entry['mean_score']:.3f}")

    display(Image("/kaggle/working/reports/curves.png"))
    display(Image("/kaggle/working/reports/confusion_matrix.png"))
else:
    skipped("xem kết quả — phiên này chưa huấn luyện")

## B4. Thử trên file bất kỳ + lưu mô hình

In [ ]:
import glob

if DO_TRAIN:
    # Bất kỳ engine nào có trong corpus — cứng nhắc "piper" là rỗng khi TTS tắt.
    mau = sorted(glob.glob("/kaggle/working/corpus/fake/*/*/*.wav"))[:5]
    mau += sorted(glob.glob("/kaggle/working/corpus/real/*/*/*.wav"))[:5]
    run("detect", *mau)
else:
    skipped("thử detect — phiên này chưa huấn luyện mô hình nào")

In [ ]:
import shutil
from pathlib import Path

# `!ls` là magic của IPython nên không lồng vào `if` được — liệt kê bằng Python.
if DO_TRAIN:
    shutil.make_archive("/kaggle/working/model",          "zip", "/kaggle/working/checkpoints")
    shutil.make_archive("/kaggle/working/reports_bundle", "zip", "/kaggle/working/reports")
    for _zip in sorted(Path("/kaggle/working").glob("*.zip")):
        print(f"{_zip.stat().st_size / 1024**2:8.1f} MB  {_zip}")
else:
    skipped("đóng gói mô hình")

---
### Vài nút chỉnh hay dùng

```python
# Đổi backbone (cache đặc trưng tách riêng nên không đụng nhau)
run("run", "features", "train", "evaluate", "--set", "features.backbone.name=wav2vec2")

# Đo khả năng tổng quát sang engine chưa từng thấy
run("split", "--holdout", "omnivoice")
run("run", "features", "train", "evaluate")

# Augment mạnh tay hơn nếu clean và augmented chênh lệch nhiều
run("augment", "--copies", 3, "--set", "augment.ops.codec.p=0.8")
```

Toàn bộ tham số nằm trong `configs/default.yaml` (bản Kaggle kế thừa nó qua
`configs/kaggle.yaml`) — xem bằng `!cat configs/default.yaml`.